### SEN163B Responsible Data Analytics - Group 5 project
Group project (5 students): predictive modelling on a Dutch household panel dataset (MPN). Willem Elshout was one of the authors.

In [ ]:
pip install pandas pyreadstat matplotlib seaborn scikit-learn statsmodels shap xgboost lightgbm shap

In [ ]:
import pandas as pd
import numpy as np
import os
import time
import pyreadstat
import shap

# Statistical and model evaluation
import scipy.stats as stats
from scipy.stats import zscore, chi2_contingency
import statsmodels.api as sm
from sklearn.model_selection import KFold, cross_val_score, train_test_split, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score

# Preprocessing
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, LabelEncoder

# Machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC

# Visualization
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import seaborn as sns

# Utility
from collections import Counter


# Loading the correct dataset
Make sure to have the file "cleaned_data2.0.sav" in your folder with the notebook

In [ ]:
# Get the current working directory (the folder where the notebook is)
current_directory = os.getcwd()

# Define the file name
file_name = 'cleaned_data2.0.sav'

# Construct the full file path
file_path = os.path.join(current_directory, file_name)

# Load the file (assuming it's a CSV, adjust if it's a different format)
final_data, meta_1 = pyreadstat.read_sav(file_path)

# Display the first few rows of the data to check
final_data.describe()


In [ ]:
# Local paths (hardcoded in the code just to let the cells from Part I run. The actual analysis and model building will be done on the cleaned data, which is not hardcoded)

dagboek_data, meta_1 = pyreadstat.read_sav("data/MPNWAVE10_DAGBOEKdata_en.sav")
hh_data, meta_2 = pyreadstat.read_sav("data/MPNWAVE10_HHdata_en.sav")
p_data, meta_3 = pyreadstat.read_sav("data/MPNWAVE10_Pdata_en.sav")
p_data_bijzonder, meta_4 = pyreadstat.read_sav("data/MPNWAVE10_Pdata bijzonder_en.sav")
dag_data, meta_5 = pyreadstat.read_sav("data/MPNWAVE10_DAGdata_en.sav")


# Part I
## Cleaning and combining the data

 Selecting the variables for DAGBOEKdata.sav

In [ ]:
# Updated list of variables to keep
variables_to_keep = ["HHID", "PERSID", "KHVM", "KMOTIEF", "AFSTV", "WOONPC2"]
dagboek_data_keep = dagboek_data[variables_to_keep]

# Display dataset info
dagboek_data_keep.info()

 Selecting the variables for HHdata.sav

In [ ]:
# Updated list of variables to keep
variables_to_keep = ["HHID", "HHBRUTOINK2_w5", "HHAUTO_N"]
hh_data_keep = hh_data[variables_to_keep]

# Display dataset info
hh_data_keep.info()

 Selecting the variables for Pdata.sav

In [ ]:
# Updated list of variables to keep
variables_to_keep = [
    "PERSID", "KLEEFT2", "GESLACHT", "HERKOMST_w5",
    "OPLEIDING", "PAUTO", "PFIETS", "PHYBRID", "PEBIKE",
    "N_URENVAST", "N_URENTHUIS", "N_URENELDERS", "N_URENONDERWEG"
]

# Filter the dataset
p_data_keep = p_data[variables_to_keep]

# Display dataset info
p_data_keep.info()


Merging the datasets together

In [ ]:
# Merge first two datasets on "PERSID"
merged_data = pd.merge(p_data_keep, dagboek_data_keep, on="PERSID", how="inner")

# Merge the final dataset on "HHID"
final_merged_data = pd.merge(merged_data, hh_data_keep, on="HHID", how="inner")

# Save the merged dataset in a .sav file
output_file = "final_merged_data.sav"

# Save the DataFrame to a .sav file
# pyreadstat.write_sav(final_merged_data, output_file)

# Display dataset info
final_merged_data.info()

Filling empty cells in AFSTV to 0

In [ ]:
# Replace NaN in 'AFSTV' with 0
final_merged_data['AFSTV'] = final_merged_data['AFSTV'].fillna(0)

# Drop NaN values for all other variables
cleaned_data = final_merged_data.dropna()

# Display dataset info
cleaned_data.info()

# Save the cleaned dataset as an SPSS (.sav) file
# pyreadstat.write_sav(cleaned_data, "cleaned_data.sav")

Removal of irrelevant values

In [ ]:
# List of columns where 99 needs to be removed
columns_to_remove_99 = [
    'N_URENVAST', 'N_URENTHUIS', 'N_URENELDERS', 'N_URENONDERWEG',
    'PAUTO', 'PFIETS', 'PHYBRID', 'PEBIKE', 'HHAUTO_N'
]

# Remove rows where these columns contain 99
for col in columns_to_remove_99:
    cleaned_data = cleaned_data.loc[cleaned_data[col] != 99]

# Remove rows where 'OPLEIDING' contains 9, 99, or 999
cleaned_data = cleaned_data.loc[~cleaned_data['OPLEIDING'].isin([9, 99, 999])]

# Remove rows where 'HERKOMST_w5' contains 4 or 99
cleaned_data = cleaned_data.loc[~cleaned_data['HERKOMST_w5'].isin([4, 99])]

# Remove rows where 'HHBRUTOINK2_w5' contains 7
cleaned_data = cleaned_data.loc[cleaned_data['HHBRUTOINK2_w5'] != 7]

# Remove rows where 'KHVM' contains 5, 8
cleaned_data = cleaned_data.loc[~cleaned_data['KHVM'].isin([5, 8])]

# Remove rows where 'KMOTIEF' contains 9 or 10
cleaned_data = cleaned_data.loc[~cleaned_data['KMOTIEF'].isin([9, 10])]

# Remove rows where 'KLEEFT2' contains 1 or 2
cleaned_data = cleaned_data.loc[~cleaned_data['KLEEFT2'].isin([1, 2])]

# Save the cleaned dataset as an SPSS (.sav) file
# pyreadstat.write_sav(cleaned_data, "cleaned_data_removal.sav")

# Display dataset info
cleaned_data.info()


Recategorisation of the categorial variables to reduce the model size and improve interpretation

In [ ]:
# List of columns to recategorize (N_UREN* variables)
columns_to_recategorize = ['N_URENVAST', 'N_URENTHUIS', 'N_URENELDERS', 'N_URENONDERWEG']

# Recategorization function for N_UREN* variables
def recategorize_hours(value):
    if value == 98:
        return 1
    elif value in [1, 2, 3]:
        return 2
    elif value in [4, 5, 6]:
        return 3
    else:
        return value  # Keep all other values unchanged

# Apply recategorization to each N_UREN* column
for col in columns_to_recategorize:
    cleaned_data[col] = cleaned_data[col].apply(recategorize_hours)

# Recategorization function for HHBRUTOINK2_w5
def recategorize_income(value):
    if value in [1, 2]:
        return 1
    elif value in [3, 4]:
        return 2
    elif value in [5, 6]:
        return 3
    else:
        return value  # Keep all other values unchanged

# Apply recategorization to HHBRUTOINK2_w5
cleaned_data['HHBRUTOINK2_w5'] = cleaned_data['HHBRUTOINK2_w5'].apply(recategorize_income)

# Recategorization function for KLEEFT2
def recategorize_kleeft2(value):
    if value in [3, 4]:
        return 1
    elif value in [5]:
        return 2
    elif value in [6]:
        return 3
    elif value in [7]:
        return 4
    elif value in [8]:
        return 5
    elif value in [9, 10]:
        return 6
    else:
        return value  # Keep all other values unchanged

# Apply recategorization to KLEEFT2
cleaned_data['KLEEFT2'] = cleaned_data['KLEEFT2'].apply(recategorize_kleeft2)

# Recategorization function for OPLEIDING
def recategorize_opleiding(value):
    if value in [1, 3]:
        return 1
    elif value in [4, 5, 6]:
        return 2
    elif value in [7, 8]:
        return 3
    else:
        return value  # Keep all other values unchanged

# Apply recategorization to OPLEIDING
cleaned_data['OPLEIDING'] = cleaned_data['OPLEIDING'].apply(recategorize_opleiding)

# Recategorization function for KHVM
def recategorize_khvm(value):
    if value in [1, 2]:
        return 0
    elif value in [3, 4]:
        return 1
    elif value in [6, 7]:
        return 2
    elif value == 99:
        return 3
    else:
        return value  # Keep all other values unchanged

# Apply recategorization to KHVM
cleaned_data['KHVM'] = cleaned_data['KHVM'].apply(recategorize_khvm)

# Recategorization function for KMOTIEF
def recategorize_kmotief(value):
    if value in [1, 2]:
        return 1
    elif value in [3, 4]:
        return 2
    elif value == 5:
        return 3
    elif value in [6, 7, 8]:
        return 4
    elif value == 99:
        return 5
    else:
        return value  # Keep all other values unchanged

# Apply recategorization to KMOTIEF
cleaned_data['KMOTIEF'] = cleaned_data['KMOTIEF'].apply(recategorize_kmotief)

# Recategorization function for PAUTO, PFIETS, PEBIKE, PHYBRID
def recategorize_vehicle(value):
    if value in [0, 2]:
        return 2
    else:
        return value  # Keep all other values unchanged

# Apply recategorization to vehicle-related columns
vehicle_columns = ['PAUTO', 'PFIETS', 'PEBIKE', 'PHYBRID']
for col in vehicle_columns:
    cleaned_data[col] = cleaned_data[col].apply(recategorize_vehicle)


# Recategorization function for WOONPC2
def recategorize_woonpc2(value):
    if 10 <= value <= 39:  # If value is in the range 10-39
        return 1
    elif 40 <= value <= 69:  # If value is in the range 40-69
        return 2
    elif 70 <= value <= 99:  # If value is in the range 70-99
        return 3
    else:
        return value  # Keep all other values unchanged

# Apply recategorization to WOONPC2
cleaned_data['WOONPC2'] = cleaned_data['WOONPC2'].apply(recategorize_woonpc2)

# Save the cleaned dataset as an SPSS (.sav) file
pyreadstat.write_sav(cleaned_data, "cleaned_data_recat.sav")

# Display dataset info to confirm changes
cleaned_data.info()


Calculating the Z-score and outlier removal in numerical variables HHAUTO_N and AFSTV

In [ ]:
# # Calculate Z-scores for 'AFSTV' and 'HHAUTO_N' if they exist in cleaned_data
# if 'AFSTV' in cleaned_data.columns and 'HHAUTO_N' in cleaned_data.columns:
#     cleaned_data['AFSTV_Zscore'] = zscore(cleaned_data['AFSTV'], nan_policy='omit')
#     cleaned_data['HHAUTO_N_Zscore'] = zscore(cleaned_data['HHAUTO_N'], nan_policy='omit')



# # Print the highest and lowest Z-scores for 'AFSTV' and 'HHAUTO_N'
# print("AFSTV - Max Z-score:", cleaned_data['AFSTV_Zscore'].max())
# print("AFSTV - Min Z-score:", cleaned_data['AFSTV_Zscore'].min())
# print("HHAUTO_N - Max Z-score:", cleaned_data['HHAUTO_N_Zscore'].max())
# print("HHAUTO_N - Min Z-score:", cleaned_data['HHAUTO_N_Zscore'].min())

# # Count the number of Z-scores greater than 3 for 'AFSTV' and 'HHAUTO_N'
# afstv_zscore_above_3 = (cleaned_data['AFSTV_Zscore'] > 3).sum()
# hhauto_n_zscore_above_3 = (cleaned_data['HHAUTO_N_Zscore'] > 3).sum()

# # Display the results
# print("Number of Z-scores > 3 in AFSTV:", afstv_zscore_above_3)
# print("Number of Z-scores > 3 in HHAUTO_N:", hhauto_n_zscore_above_3)

# # Remove rows where Z-scores are greater than 3
# cleaned_data = cleaned_data[(cleaned_data['AFSTV_Zscore'] <= 3) & (cleaned_data['HHAUTO_N_Zscore'] <= 3)]

# # Save the cleaned dataset as an SPSS .sav file
# # pyreadstat.write_sav(cleaned_data, "cleaned_data2.0.sav")


In [ ]:
cleaned_data.describe()

In [ ]:
final_data.describe()

## Cramer's V correlation matrix

In [ ]:
# Function to calculate Cramér's V
def cramers_v(contingency_table):
    chi2 = chi2_contingency(contingency_table)[0]
    n = contingency_table.sum().sum()
    r, k = contingency_table.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

# Define the categorical variables
variables = [
    "N_URENVAST", "N_URENTHUIS",
    "HHBRUTOINK2_w5", "KLEEFT2", "OPLEIDING", "KMOTIEF",
    "PAUTO", "PFIETS", "PEBIKE", "PHYBRID", "GESLACHT", "HERKOMST_w5", "WOONPC2"
]

# Initialize DataFrame to store Cramér's V results
cramers_v_results = pd.DataFrame(index=variables, columns=variables, dtype=float)

# Compute Cramér's V for all pairwise combinations of categorical variables
for var1 in variables:
    for var2 in variables:
        if var1 != var2:
            contingency_table = pd.crosstab(final_data[var1], final_data[var2])
            cramers_v_results.loc[var1, var2] = cramers_v(contingency_table)
        else:
            cramers_v_results.loc[var1, var2] = np.nan

# Plot the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cramers_v_results.astype(float), annot=True, cmap="coolwarm", fmt=".2f",
            linewidths=0.5, linecolor='gray', square=True, cbar_kws={'label': "Cramér's V Value"})

plt.title("Correlation Matrix (Cramér's V Value)")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

# Show the plot
plt.show()

## Exploring the categorical variables

Visualization of the gender distribution

In [ ]:
df = pd.DataFrame(final_data)

# Map the values in the 'GESLACHT' column
df['GESLACHT'] = df['GESLACHT'].replace({1.0: 'Male', 2.0: 'Female'})

# Calculate the value counts and percentages
value_counts = df['GESLACHT'].value_counts()
percentages = (value_counts / value_counts.sum()) * 100

# Plot the histogram of the 'GESLACHT' column
plt.figure(figsize=(10, 6))
bars = value_counts.plot(kind='bar', color=['blue', 'orange'])  # Assign different colors for clarity
plt.title('Distribution of Gender')
plt.xlabel('Gender')
plt.ylabel('Frequency')

# Create custom legend labels with percentages rounded to 0 decimals
legend_labels = [f'{category}: {percentage:.0f}%' for category, percentage in percentages.items()]
print(legend_labels)

# Manually create legend patches
legend_patches = [mpatches.Patch(color=bar.get_facecolor(), label=label) for bar, label in zip(bars.patches, legend_labels)]

# Add the manually created legend
plt.legend(handles=legend_patches, title='Proportions', loc='upper right')

plt.show()

Visualization of the age distribution

In [ ]:
# Define category mapping with proper order
Mode_mapping = {
    1: "Young Adults",
    2: "Young professionals",
    3: "Middle-aged",
    4: "Late adulthood",
    5: "Seniors",
    6: "Elderly"
}

df['KLEEFT2'] = df['KLEEFT2'].replace(Mode_mapping)

# Ensure categorical type with the desired order
age_order = ["Young Adults", "Young professionals", "Middle-aged", "Late adulthood", "Seniors", "Elderly"]
df['KLEEFT2'] = pd.Categorical(df['KLEEFT2'], categories=age_order, ordered=True)

# Set plot style
sns.set_style("whitegrid")

# Count occurrences of each age category
age_counts = df["KLEEFT2"].value_counts()

# Sort by predefined category order
age_counts = age_counts.reindex(age_order)

# Plot age distribution
plt.figure(figsize=(10, 6))
sns.barplot(x=age_counts.index, y=age_counts.values, order=age_order, palette="viridis")

# Add labels and title
plt.xlabel("Age", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.title("Distribution of Age", fontsize=14)
plt.xticks(rotation=45)  # Rotate x-axis labels if necessary

# Show plot
plt.show()


Visualization of the mode of transport distribution

In [ ]:
# Define category mapping with proper order
Mode_mapping = {
    1: "Car",
    2: "Public Transport",
    3: "Active Transport",
    4: "No Trip"
}

df['KHVM'] = df['KHVM'].replace(Mode_mapping)

# Ensure categorical type with the desired order
transport_order = ["Car", "Public Transport", "Active Transport", "No Trip"]
df['KHVM'] = pd.Categorical(df['KHVM'], categories=transport_order, ordered=True)

# Set plot style
sns.set_style("whitegrid")

# Count occurrences of each transport mode
transport_counts = df["KHVM"].value_counts()

# Sort by predefined category order
transport_counts = transport_counts.reindex(transport_order)

# Plot transport mode choice
plt.figure(figsize=(10, 6))
sns.barplot(x=transport_counts.index, y=transport_counts.values, order=transport_order, palette="viridis")

# Add labels and title
plt.xlabel("Transport Mode Choice", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.title("Distribution of Transport Mode Choices", fontsize=14)
plt.xticks(rotation=45)  # Rotate x-axis labels if necessary

# Show plot
plt.show()


Visualization of income distribution

In [ ]:
# Set plot style
sns.set_style("whitegrid")

# Define income category mapping (based on provided table)
income_mapping = {
    1: "Low Income",
    2: "Medium Income",
    3: "High Income",
}

# Replace numerical values with category labels
df["Income_Category"] = df["HHBRUTOINK2_w5"].replace(income_mapping)

# Ensure categorical type with the desired order
income_order = ["Low Income", "Medium Income", "High Income"]
df["Income_Category"] = pd.Categorical(df["Income_Category"], categories=income_order, ordered=True)

# Count occurrences of each income group
income_counts = df["Income_Category"].value_counts()

# Sort by predefined category order
income_counts = income_counts.reindex(income_order)

# Plot income distribution
plt.figure(figsize=(8, 5))
sns.barplot(x=income_counts.index, y=income_counts.values, order=income_order, palette="viridis")

# Add labels and title
plt.xlabel("Income Category", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.title("Distribution of Income Categories", fontsize=14)

# Show plot
plt.show()

Visualization of working hours distribution

In [ ]:
# Set plot style
sns.set_style("whitegrid")

# Define mapping for working hours categories based on the table
working_hours_mapping = {
    1: "Unemployed",
    2: "Part-time",
    3: "Near full-time/full-time",
}

# Apply mapping to N_URENVAST
df["Working_Hours_Fixed"] = df["N_URENVAST"].replace(working_hours_mapping)

# Apply mapping to N_URENTHUIS
df["Working_Hours_Remote"] = df["N_URENTHUIS"].replace(working_hours_mapping)

# Define correct category order
working_hours_order = ["Unemployed", "Part-time", "Near full-time/full-time"]

# Convert to categorical type with proper order
df["Working_Hours_Fixed"] = pd.Categorical(df["Working_Hours_Fixed"], categories=working_hours_order, ordered=True)
df["Working_Hours_Remote"] = pd.Categorical(df["Working_Hours_Remote"], categories=working_hours_order, ordered=True)

# Count occurrences for both fixed and remote work hours
fixed_hours_counts = df["Working_Hours_Fixed"].value_counts().reindex(working_hours_order)
remote_hours_counts = df["Working_Hours_Remote"].value_counts().reindex(working_hours_order)

# Plot working hours distribution for Working hours in fixed location
plt.figure(figsize=(8,5))
sns.barplot(x=fixed_hours_counts.index, y=fixed_hours_counts.values, order=working_hours_order, palette="viridis")
plt.xlabel("Working hours in fixed location category", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.title("Distribution Working hours in fixed location", fontsize=14)
plt.show()

# Plot working hours distribution for remote work
plt.figure(figsize=(8,5))
sns.barplot(x=remote_hours_counts.index, y=remote_hours_counts.values, order=working_hours_order, palette="viridis")
plt.xlabel("Remote Working Hours Category", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.title("Distribution of Remote Working Hours", fontsize=14)
plt.show()


Visualization of Age vs Mode choice

In [ ]:
# Define category mapping with proper order
Mode_mapping = {
    1: "Young Adults",
    2: "Young professionals",
    3: "Middle-aged",
    4: "Late adulthood",
    5: "Seniors",
    6: "Elderly"
}

df['KLEEFT2'] = df['KLEEFT2'].replace(Mode_mapping)

# Ensure categorical type with the desired order
age_order = ["Young Adults", "Young professionals", "Middle-aged", "Late adulthood", "Seniors", "Elderly"]
df['KLEEFT2'] = pd.Categorical(df['KLEEFT2'], categories=age_order, ordered=True)

# Set plot style
sns.set_style("whitegrid")

# Create a crosstab of KLEEFT2 and KHVM
crosstab = pd.crosstab(df['KLEEFT2'], df['KHVM'])

# Plot the stacked bar chart
crosstab.plot(kind='bar', stacked=True, figsize=(12, 8), colormap='viridis')

# Add labels and title
plt.title('Stacked Bar Chart of Age vs Mode of Transport', fontsize=14)
plt.xlabel('Age Groups', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.legend(title='Mode of Transport', fontsize=10)

# Show the plot
plt.tight_layout()
plt.show()

Visualization of Income vs Mode choice

In [ ]:
# Set plot style
sns.set_style("whitegrid")

# Ensure the columns 'HHBRUTOINK2_w5' and 'KHVM' exist in the dataframe
if 'HHBRUTOINK2_w5' in df.columns and 'KHVM' in df.columns:
    # Bin the household income into categories for better visualization
    df['Income_Bins'] = pd.cut(df['HHBRUTOINK2_w5'], bins=5, labels=["Low", "Lower-Mid", "Mid", "Upper-Mid", "High"])

    # Create a crosstab of Income_Bins and KHVM
    crosstab = pd.crosstab(df['Income_Bins'], df['KHVM'])

    # Plot the stacked bar chart
    crosstab.plot(kind='bar', stacked=True, figsize=(12, 8), colormap='viridis')

    # Add labels and title
    plt.title('Stacked Bar Chart of Household Income vs Mode of Transport', fontsize=14)
    plt.xlabel('Household Income Categories', fontsize=12)
    plt.ylabel('Count', fontsize=12)
    plt.legend(title='Mode of Transport (KHVM)', fontsize=10)

    # Show the plot
    plt.tight_layout()
    plt.show()
else:
    print("Columns 'HHBRUTOINK2_w5' and/or 'KHVM' are not available in the dataframe.")

# Part II
## Building the model

In [ ]:
df = final_data
df.describe()

### Split the data and preparing the model

In [ ]:
# Define the dependent variable (target) for prediction
y = df["KHVM"]

# Define the list of categorical features to be one-hot encoded
categorical_vars = [
    "N_URENVAST", "N_URENTHUIS", "HHBRUTOINK2_w5", "KLEEFT2", "OPLEIDING",
    "KMOTIEF", "PAUTO", "PFIETS", "PEBIKE", "PHYBRID",
    "GESLACHT", "HERKOMST_w5", "WOONPC2"
]

# Define the list of numerical features
numerical_vars = [
    "AFSTV", "HHAUTO_N"
]

# Shuffle the dataset to remove any unintended ordering (e.g., based on time or geography)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# One-hot encode categorical variables, excluding the first level to avoid multicollinearity
encoder = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
X_categorical = encoder.fit_transform(df[categorical_vars])
X_categorical_df = pd.DataFrame(X_categorical, columns=encoder.get_feature_names_out(categorical_vars))

# Standardize numerical variables to mean 0 and standard deviation 1
scaler = StandardScaler()
X_numerical = scaler.fit_transform(df[numerical_vars])
X_numerical_df = pd.DataFrame(X_numerical, columns=numerical_vars)

# Combine encoded categorical variables and scaled numerical variables into a single feature matrix
X = pd.concat([X_categorical_df, X_numerical_df], axis=1)

# Redefine the target (y) after processing (to ensure alignment with shuffled df)
y = df["KHVM"]

# Drop features with zero variance
X = X.loc[:, X.std() != 0]

# Drop any columns that contain NaN or infinite values (infinite values are replaced with NaNs first)
X = X.replace([np.inf, -np.inf], np.nan).dropna(axis=1)

# Split the dataset into training (70%) and temporary (30%) sets, stratified to maintain class balance
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Split the temporary set into validation (15%) and test (15%) sets
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)


### Checking values within the model

In [ ]:
# print("Any NaNs?", X_train.isnull().any().any(), y_train.isnull().any())
# print("Any Infs?", np.isinf(X_train.values).any())
# print("All finite?", np.isfinite(X_train.values).all())
# print("X_train shape:", X_train.shape)
# print("y_train unique:", y_train.unique())
# print("Max value in X_train:", X_train.max().max())
# print("Min value in X_train:", X_train.min().min())

## Building a Multinomial Logistic Regression model 

In [ ]:
# Fit the Logistic Regression model.
# 'max_iter=1000' increases the number of iterations to ensure convergence.
model_mnl  = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
model_mnl.fit(X_train, y_train)

# Make predictions on the training, validation, and test sets.
y_train_pred = model_mnl.predict(X_train)
y_val_pred = model_mnl.predict(X_val)
y_test_pred = model_mnl.predict(X_test)
print(y_test_pred.shape)
# Evaluate and print accuracy scores for each set.
print("Training Accuracy:", accuracy_score(y_train, y_train_pred))
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))

# Print a detailed classification report on the test set:
# Includes precision, recall, F1-score, and support for each class.
print("\nClassification Report Logistic Regression:")
print(classification_report(y_test, y_test_pred))

# Print the confusion matrix to visualize the distribution of correct and incorrect predictions.
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

## Building a Random Forest Classifier 

In [ ]:
# Initialize a Random Forest classifier with balanced class weights to handle class imbalance.
# 'class_weight="balanced"' automatically adjusts weights inversely proportional to class frequencies.
model_rfc  = RandomForestClassifier(random_state=42, class_weight='balanced')

# Train the Random Forest model on the training data.
model_rfc.fit(X_train, y_train)

# Use the trained model to predict labels for the test set.
y_pred_rfc = model_rfc.predict(X_test)

# Print a detailed classification report (precision, recall, F1-score, and support per class).
print("RandomForest Balanced Classification Report:")
print(classification_report(y_test, y_pred_rfc))

# Print the confusion matrix to evaluate how well the model classified each class.
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rfc))

## Building an XGBoost model

In [ ]:

# Encode the labels (y) as integer values starting from 0.
# This is necessary because XGBoost expects class labels as integers from 0 to (num_class - 1).
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split the dataset into training and temporary sets (70% training, 30% temporary)
# Stratified splitting ensures that the class distribution is maintained in each split.
X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.3, stratify=y_encoded, random_state=42)

# Split the temporary set evenly into validation and test sets (15% each of the total data)
# Again using stratification to maintain class balance.
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# Initialize the XGBoost classifier for multi-class classification.
xgb_model = XGBClassifier(
    objective='multi:softmax',
    num_class=len(np.unique(y_train)),
    eval_metric='mlogloss',
     random_state=42
)

# Train the XGBoost classifier on the training set.
xgb_model.fit(X_train, y_train)

# Predict class labels on the test set.
xgb_preds = xgb_model.predict(X_test)

# Print the classification report, which includes precision, recall, and F1-score for each class.
print("XGBoost Classification Report:")
print(classification_report(y_test, xgb_preds))

# Print the confusion matrix to show how well the model distinguished between classes.
print("Confusion Matrix:")
print(confusion_matrix(y_test, xgb_preds))


## Cross validation

### Random Forest Classifier

In [ ]:

# Assuming final_data is already loaded as a pandas DataFrame
# Separating features and target variable
X2 = final_data.iloc[:, :-1]  # All columns except the last one as features
y2 = final_data.iloc[:, -1]   # The last column as the target variable

# Encode target variable if it's categorical
y2 = LabelEncoder().fit_transform(y2)

# Initialize model (RandomForestClassifier as an example)
model_rfc2 = RandomForestClassifier(random_state=42)

# Perform 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model_rfc2, X2, y2, cv=kf, scoring='accuracy')

# Print results
print(f'Cross-validation scores: {scores}')
print(f'Average accuracy: {np.mean(scores):.4f}')
print(f'Standard deviation: {np.std(scores):.4f}')

### XGBoost model

In [ ]:
# Assuming final_data is already loaded as a pandas DataFrame
# Separating features and target variable
X3 = final_data.iloc[:, :-1]  # ALL columns except the last one as features
y3 = final_data.iloc[:, -1]   # The last column as the target variable

# Encode target variable if it's categorical
y3 = LabelEncoder().fit_transform(y3)

# Initialize XGBoost model
model_xgb = XGBClassifier(random_state=42, eval_metric='logloss')

# Perform 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model_xgb, X3, y3, cv=kf, scoring='accuracy')

# Print results
print(f'Cross-validation scores: {scores}')
print(f'Average accuracy: {np.mean(scores):.4f}')
print(f'Standard deviation: {np.std(scores):.4f}')

### Multinomial Logistic Regression model

In [ ]:
# Assuming final_data is already loaded as a pandas DataFrame
# Separating features and target variable
X4 = final_data.iloc[:, :-1]  # ALL columns except the last one as features
y4 = final_data.iloc[:, -1]   # The last column as the target variable

# Encode target variable if it's categorical
y4 = LabelEncoder().fit_transform(y4)

# Initialize Logistic Regression model
model_mnl = LogisticRegression(random_state=42, max_iter=1000)

# Perform 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model_mnl, X4, y4, cv=kf, scoring='accuracy')

# Print results
print(f'Cross-validation scores: {scores}')
print(f'Average accuracy: {np.mean(scores):.4f}')
print(f'Standard deviation: {np.std(scores):.4f}')

## Ensemble predictor

In [ ]:
# Ensemble prediction
start_time = time.time()
estimators = [('mnl', model_mnl), ('rf', model_rfc), ('xgb', xgb_model)]
ens = VotingClassifier(estimators, voting='soft')
ens.fit(X_train, y_train.ravel())
print('Ensemble scores:', ens.score(X_test, y_test.ravel()))
print('Time to compute:', round(time.time() - start_time, 2))

### Ensemble prediction with cross-validation

In [ ]:
# Create the ensemble model
estimators = [('mml', model_mnl), ('rf', model_rfc), ('xgb', xgb_model)]
ens = VotingClassifier(estimators, voting='soft')

# Perform 5-fold cross-validation
cv_scores = cross_val_score(ens, X_train, y_train.ravel(), cv=5, scoring='accuracy')

# Fit the ensemble model on the full training set
ens.fit(X_train, y_train.ravel())

# Print the cross-validation scores and mean score
print('Cross-validation scores:', cv_scores)
print('Mean cross-validation score:', cv_scores.mean())

# Print the test score
print('Ensemble test score:', ens.score(X_test, y_test.ravel()))

### Ensemble prediction with cross-validation and false positive rate

In [ ]:

# Create the ensemble model
estimators = [('mml', model_mnl), ('rf', model_rfc), ('xgb', xgb_model)]
ens = VotingClassifier(estimators, voting='soft')

# Perform 5-fold cross-validation and get predictions
y_train_pred = cross_val_predict(ens, X_train, y_train.ravel(), cv=5)

# Calculate cross-validation metrics
cv_accuracy = accuracy_score(y_train.ravel(), y_train_pred)
cv_cm = confusion_matrix(y_train.ravel(), y_train_pred)
cv_fp = cv_cm[0, 1]  # False positives
cv_total_negatives = cv_cm[0, 0] + cv_cm[0, 1]  # Actual negatives
cv_fpr = (cv_fp / cv_total_negatives) * 100  # False positive rate in percentage

# Fit the ensemble model on the full training set
ens.fit(X_train, y_train.ravel())

# Get test set predictions and metrics
y_test_pred = ens.predict(X_test)
test_accuracy = ens.score(X_test, y_test.ravel())
test_cm = confusion_matrix(y_test.ravel(), y_test_pred)
test_fp = test_cm[0, 1]  # False positives
test_total_negatives = test_cm[0, 0] + test_cm[0, 1]  # Actual negatives
test_fpr = (test_fp / test_total_negatives) * 100  # False positive rate in percentage

# Print results
print("Cross-validation results:")
print(f"Accuracy: {cv_accuracy:.4f}")
print(f"False positive rate: {cv_fpr:.2f}%")
print(f"Confusion matrix:\n{cv_cm}\n")

print("Test set results:")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"False positive rate: {test_fpr:.2f}%")
print(f"Confusion matrix:\n{test_cm}")

## Bias and Fairness Analysis

In [ ]:
# Ethnicity label mapping
ethnicity_map = {
    1: "Native Dutch",
    2: "European (non-Dutch)",
    3: "Non-European",
}

# Apply mapping
df["HERKOMST_w5"] = df["HERKOMST_w5"].replace(ethnicity_map)
# Optional: Transport mode mapping (if still numeric)
# Create a dictionary that maps numeric values (1, 2, 3, 4) to descriptive transport modes
mode_mapping = {
    1: "Car",                
    2: "Public Transport",   
    3: "Active Transport",   
    4: "No Trip"          }   

# Replace the numeric values in the 'KHVM' column of the dataframe with the corresponding transport mode names
df["KHVM"] = df["KHVM"].replace(mode_mapping)

# Set up the size of the plot (12x6 inches)
plt.figure(figsize=(12,6))

# Create a count plot (bar chart) with 'HERKOMST_w5' on the x-axis and 'KHVM' (transport mode) as the hue (color distinction)
sns.countplot(data=df, x="HERKOMST_w5", hue="KHVM", palette="Set2")

plt.title("Transport Mode Choice by Ethnicity", fontsize=14)
plt.xlabel("Ethnicity", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.legend(title="Transport Mode")
plt.xticks(rotation=45)
plt.tight_layout()

# Display the plot
plt.show()


### Visualization of Ethnicity groups and their mode choices

In [ ]:
# Optional: Map transport modes if needed
mode_mapping = {
    1: "Car",
    2: "Public Transport",
    3: "Active Transport",
    4: "No Trip"
}
df["KHVM"] = df["KHVM"].replace(mode_mapping)

# Ethnicity label mapping (only the 3 relevant groups)
ethnicity_map = {
    1: "Native Dutch",
    2: "European (non-Dutch)",
    3: "Non-European"
}
df["HERKOMST_w5"] = df["HERKOMST_w5"].replace(ethnicity_map)

# Filter dataset to include only relevant ethnicities
filtered_df = df[df["HERKOMST_w5"].isin(ethnicity_map.values())]

# Create a figure with subplots — one per ethnicity group
ethnicities = filtered_df["HERKOMST_w5"].unique()
n = len(ethnicities)

fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))

for i, eth in enumerate(ethnicities):
    ax = axes[i] if n > 1 else axes
    data = filtered_df[filtered_df["HERKOMST_w5"] == eth]["KHVM"].value_counts()
    ax.pie(data.values, labels=data.index, autopct='%1.1f%%', startangle=90, colors=plt.cm.Set3.colors)
    ax.set_title(f"Transport Mode for {eth}")

plt.tight_layout()
plt.show()


### Relative distribution of Ethnicity

In [ ]:
# Ethnicity mapping
ethnicity_map = {
    1: "Native Dutch",
    2: "European (non-Dutch)",
    3: "Non-European",
}
df["HERKOMST_w5"] = df["HERKOMST_w5"].replace(ethnicity_map)

# Calculate relative values
ethnicity_rel = df["HERKOMST_w5"].value_counts(normalize=True) * 100

# Plot pie chart
plt.figure(figsize=(8, 8))
plt.pie(ethnicity_rel.values,
        labels=ethnicity_rel.index,
        autopct='%1.1f%%',
        startangle=90,
        colors=plt.cm.Pastel1.colors)

plt.title("Relative Distribution of Ethnicity", fontsize=14)
plt.tight_layout()
plt.show()

## Transparancy and Explainability

### SHAP analysis

In [ ]:
# 1. Maak de SHAP explainer aan voor je Random Forest
explainer = shap.TreeExplainer(model_rfc)

X_test_sample = X_test.sample(n=100, random_state=42)
# 2. Bereken SHAP waarden voor je testsetb
shap_values = explainer(X_test_sample)

# 3. Plot een samenvatting van de belangrijkste features
# Plot belangrijkste features
shap.summary_plot(shap_values, X_test_sample, plot_type="bar", max_display=10)

### Manual Fariness Check
(Run RFC model before running this)

In [ ]:
# Get Gender
X_test_copy = X_test.copy()
X_test_copy["gender"] = final_data.loc[X_test.index, "GESLACHT"].astype(str)
X_test_copy["y_true"] = y_test
X_test_copy["y_pred"] = y_pred_rfc

# Fairness check per group
for group in X_test_copy["gender"].unique():
    group_data = X_test_copy[X_test_copy["gender"] == group]
    
    acc = accuracy_score(group_data["y_true"], group_data["y_pred"])
    prec = precision_score(group_data["y_true"], group_data["y_pred"], average="weighted")
    rec = recall_score(group_data["y_true"], group_data["y_pred"], average="weighted")
    
    print(f"--- Gender: {group} ---")
    print(f"Accuracy : {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall   : {rec:.3f}")


In [ ]:
# Add Ethnicity to the test set predictions
X_test_copy = X_test.copy()
X_test_copy["ethnicity"] = final_data.loc[X_test.index, "HERKOMST_w5"]
X_test_copy["y_true"] = y_test
X_test_copy["y_pred"] = y_pred_rfc

ethnicity_map = {
    1: "Native Dutch",
    2: "European (non-Dutch)",
    3: "Non-European",
}

X_test_copy["ethnicity"] = X_test_copy["ethnicity"].replace(ethnicity_map)

# Filter for relevant groups (1, 2, 3)
relevant_groups = ["Native Dutch", "European (non-Dutch)", "Non-European"]
filtered_test = X_test_copy[X_test_copy["ethnicity"].isin(relevant_groups)]

# Fairness check per ethnicity group
for group in filtered_test["ethnicity"].unique():
    group_data = filtered_test[filtered_test["ethnicity"] == group]
    
    acc = accuracy_score(group_data["y_true"], group_data["y_pred"])
    prec = precision_score(group_data["y_true"], group_data["y_pred"], average="macro")
    rec = recall_score(group_data["y_true"], group_data["y_pred"], average="macro")
    
    print(f"\n--- Ethnicity: {group} ---")
    print(f"Accuracy : {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall   : {rec:.3f}")